In [0]:
# %pip install databricks-feature-engineering
%pip install databricks-feature-engineering "protobuf>=5.29.4"

In [0]:
 %restart_python

In [0]:
import logging
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Config
CATALOG = "oil_gas_demo"
SOURCE_TABLE = f"{CATALOG}.gold.demand_features"
FEATURE_TABLE = f"{CATALOG}.gold.demand_feature_store"
LABEL_TABLE = f"{CATALOG}.gold.demand_labels"

fe = FeatureEngineeringClient()


class DemandFeatureStore:
    """Registers features in Feature Store and saves labels separately."""

    def __init__(self, spark, source_table, feature_table, label_table):
        self.spark = spark
        self.source_table = source_table
        self.feature_table = feature_table
        self.label_table = label_table
        self.data = None

    def read_gold_table(self):
        """Step 1: Load the gold feature table."""
        logger.info(f"Loading: {self.source_table}")
        self.data = self.spark.table(self.source_table)
        logger.info(f"Loaded {self.data.count():,} rows")

    def add_primary_key(self):
        """Step 2: Add feature_uid as primary key."""
        logger.info("Adding primary key: feature_uid")
        self.data = self.data.withColumns({
            "feature_uid": F.concat_ws(
                "_",
                F.col("transaction_date").cast("string"),
                F.col("product_name"),
                F.col("destination_city"),
            ),
        })
        logger.info("Primary key added: feature_uid")

    def register_feature_table(self):
        """Step 3: Register features (no label) in Feature Store."""
        feature_data = self.data.select(
            "feature_uid", "transaction_date", "product_name", "destination_city",
            "avg_unit_price", "total_inventory",
            "day_of_week", "month",
            "demand_lag_1", "demand_lag_7", "rolling_avg_7",
        )

        if not self.spark.catalog.tableExists(self.feature_table):
            logger.info(f"Creating feature table: {self.feature_table}")
            fe.create_table(
                name=self.feature_table,
                primary_keys=["feature_uid"],
                df=feature_data,
                description="Features for demand forecasting. Label stored separately.",
                tags={
                    "name": "daily_demand_features",
                    "domain": "oil&gas",
                    "key_columns": "feature_uid, transaction_date, product_name, destination_city",
                    "categorical_columns": "product_name, destination_city",
                    "numeric_columns": "total_inventory, day_of_week, month, demand_lag_1, demand_lag_7, rolling_avg_7",
                },
            )
            logger.info("Feature table created.")
        else:
            logger.info(f"Feature table already exists: {self.feature_table}")

        logger.info(f"Writing features via merge: {self.feature_table}")
        fe.write_table(name=self.feature_table, df=feature_data, mode="merge")
        logger.info(f"Features written ({feature_data.count():,} rows)")

    def save_label_data(self):
        """Step 4: Save total_demand as the label."""
        label_data = self.data.select("feature_uid", "total_demand")

        logger.info(f"Writing labels to: {self.label_table}")
        label_data.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(self.label_table)
        logger.info(f"Labels written ({label_data.count():,} rows)")

    def run(self):
        """Read -> add key -> register features -> save labels."""
        self.read_gold_table()
        self.add_primary_key()
        self.register_feature_table()
        self.save_label_data()


# Execute
fs = DemandFeatureStore(
    spark=spark,
    source_table=SOURCE_TABLE,
    feature_table=FEATURE_TABLE,
    label_table=LABEL_TABLE,
)
fs.run()